# Model Experimentation Notebook

This notebook is for experimenting with different models and configurations for code generation.

In [ ]:
# Import necessary libraries
import sys
import os
import asyncio
import time
from typing import Dict, List

sys.path.append('..')

from src.llm.utils import create_gemini_client
from src.utils.token_counter import TokenCounter
from config import get_model_config

## Model Configuration Testing

In [ ]:
# Load and display model configuration
config = get_model_config()
print("Current Model Configuration:")
print(f"Model: {config['models']['gemini']['name']}")
print(f"Temperature: {config['models']['gemini'].get('temperature', 'Not set')}")
print(f"Max Tokens: {config['models']['gemini'].get('max_tokens', 'Not set')}")
print(f"Thinking Budget: {config['models']['gemini'].get('thinking_budget', 'Not set')}")

## Temperature Experimentation

In [ ]:
async def test_temperature_settings():
    """Test different temperature settings for creativity vs consistency."""
    
    temperatures = [0.1, 0.5, 0.7, 1.0]
    prompt = "Generate a Python function to reverse a string"
    
    results = {}
    
    for temp in temperatures:
        print(f"\nTesting temperature: {temp}")
        print("-" * 30)
        
        try:
            # Create client with specific temperature
            client = create_gemini_client(temperature=temp)
            
            # Generate response
            start_time = time.time()
            response = await client.ainvoke(prompt)
            end_time = time.time()
            
            results[temp] = {
                'response': response.content[:200] + "...",  # First 200 chars
                'response_time': end_time - start_time,
                'length': len(response.content)
            }
            
            print(f"Response time: {results[temp]['response_time']:.2f}s")
            print(f"Response length: {results[temp]['length']} chars")
            print(f"Preview: {results[temp]['response']}")
            
        except Exception as e:
            print(f"Error with temperature {temp}: {e}")
            results[temp] = {'error': str(e)}
    
    return results

# Run temperature experiment
# temp_results = await test_temperature_settings()
print("Temperature experiment ready to run. Uncomment the line above to execute.")

## Prompt Variation Testing

In [ ]:
def create_prompt_variations(base_task: str) -> List[str]:
    """Create different prompt variations for the same task."""
    
    variations = [
        # Basic prompt
        f"{base_task}",
        
        # Detailed prompt
        f"Please {base_task.lower()}. Include proper error handling, documentation, and follow best practices.",
        
        # Step-by-step prompt
        f"I need you to {base_task.lower()}. Please follow these steps:\n1. Analyze the requirements\n2. Design the solution\n3. Implement the code\n4. Add error handling",
        
        # Example-driven prompt
        f"{base_task}. Here's an example of good code structure:\n```python\ndef example_func(param):\n    \"\"\"Docstring\"\"\"\n    # Implementation\n    return result\n```\nPlease follow this pattern.",
        
        # Constraint-based prompt
        f"{base_task}. Requirements:\n- Use type hints\n- Include docstrings\n- Handle edge cases\n- Keep it under 20 lines"
    ]
    
    return variations

# Test prompt variations
base_task = "Create a Python function to find the maximum element in a list"
variations = create_prompt_variations(base_task)

print("Prompt Variations:")
for i, variation in enumerate(variations, 1):
    print(f"\n{i}. {variation[:100]}{'...' if len(variation) > 100 else ''}")

## Response Quality Comparison

In [ ]:
def evaluate_response_quality(response: str) -> Dict[str, float]:
    """Evaluate the quality of a code generation response."""
    
    import re
    
    # Quality metrics
    metrics = {
        'has_code_block': 1.0 if '```' in response else 0.0,
        'has_docstring': 1.0 if '"""' in response or "'''" in response else 0.0,
        'has_type_hints': 1.0 if re.search(r':\s*\w+', response) else 0.0,
        'has_error_handling': 1.0 if any(word in response.lower() for word in ['try', 'except', 'raise', 'assert']) else 0.0,
        'has_comments': 1.0 if '#' in response else 0.0,
        'code_length': len(re.findall(r'```[\s\S]*?```', response)[0] if '```' in response else ''),
        'explanation_length': len(response) - len(re.findall(r'```[\s\S]*?```', response)[0] if '```' in response else 0)
    }
    
    # Calculate overall quality score
    quality_score = sum([
        metrics['has_code_block'] * 0.3,
        metrics['has_docstring'] * 0.2,
        metrics['has_type_hints'] * 0.15,
        metrics['has_error_handling'] * 0.15,
        metrics['has_comments'] * 0.1,
        min(1.0, metrics['code_length'] / 200) * 0.1  # Normalize code length
    ])
    
    metrics['overall_quality'] = quality_score
    
    return metrics

# Example quality evaluation
sample_response = '''
Here's a Python function to find the maximum element in a list:

```python
def find_max(numbers: list) -> int:
    """Find the maximum element in a list of numbers.
    
    Args:
        numbers: List of numbers
        
    Returns:
        Maximum number in the list
        
    Raises:
        ValueError: If the list is empty
    """
    if not numbers:
        raise ValueError("Cannot find max of empty list")
    
    max_val = numbers[0]  # Initialize with first element
    
    for num in numbers[1:]:
        if num > max_val:
            max_val = num
    
    return max_val
```

This function handles edge cases and includes proper documentation.
'''

quality_metrics = evaluate_response_quality(sample_response)
print("Quality Evaluation:")
for metric, score in quality_metrics.items():
    if isinstance(score, float) and metric != 'code_length' and metric != 'explanation_length':
        print(f"{metric}: {score:.2f}")
    else:
        print(f"{metric}: {score}")

## Model Performance Benchmarking

In [ ]:
async def benchmark_model_performance():
    """Benchmark model performance across different types of tasks."""
    
    test_cases = [
        {
            'name': 'Simple Function',
            'prompt': 'Create a Python function to add two numbers',
            'expected_complexity': 'low'
        },
        {
            'name': 'Data Structure',
            'prompt': 'Implement a binary search tree class in Python',
            'expected_complexity': 'medium'
        },
        {
            'name': 'Algorithm',
            'prompt': 'Implement quicksort algorithm with optimization',
            'expected_complexity': 'high'
        },
        {
            'name': 'Web Scraper',
            'prompt': 'Create a web scraper to extract product data from e-commerce sites',
            'expected_complexity': 'high'
        }
    ]
    
    results = []
    token_counter = TokenCounter()
    
    for test_case in test_cases:
        print(f"\nTesting: {test_case['name']}")
        print(f"Complexity: {test_case['expected_complexity']}")
        print("-" * 40)
        
        try:
            client = create_gemini_client()
            
            start_time = time.time()
            
            # Estimate input tokens
            input_tokens = token_counter.estimate_tokens(test_case['prompt'])
            
            # Generate response (commented out for demo)
            # response = await client.ainvoke(test_case['prompt'])
            # output_tokens = token_counter.estimate_tokens(response.content)
            
            end_time = time.time()
            
            # Simulate response for demo
            response_time = end_time - start_time
            output_tokens = input_tokens * 2  # Simulated
            
            result = {
                'test_case': test_case['name'],
                'complexity': test_case['expected_complexity'],
                'response_time': response_time,
                'input_tokens': input_tokens,
                'output_tokens': output_tokens,
                'total_tokens': input_tokens + output_tokens
            }
            
            results.append(result)
            
            print(f"Response time: {response_time:.2f}s")
            print(f"Input tokens: {input_tokens}")
            print(f"Output tokens: {output_tokens}")
            
        except Exception as e:
            print(f"Error: {e}")
    
    return results

# Prepare benchmark (uncomment to run)
# benchmark_results = await benchmark_model_performance()
print("Benchmark ready to run. Uncomment the line above to execute.")

## Configuration Optimization

In [ ]:
def suggest_optimal_config(task_complexity: str) -> Dict[str, any]:
    """Suggest optimal configuration based on task complexity."""
    
    configs = {
        'low': {
            'temperature': 0.3,
            'max_tokens': 1000,
            'thinking_budget': 100,
            'description': 'Fast, consistent responses for simple tasks'
        },
        'medium': {
            'temperature': 0.5,
            'max_tokens': 2000,
            'thinking_budget': 500,
            'description': 'Balanced creativity and consistency'
        },
        'high': {
            'temperature': 0.7,
            'max_tokens': 4000,
            'thinking_budget': -1,  # Unlimited
            'description': 'Maximum creativity for complex problems'
        }
    }
    
    return configs.get(task_complexity, configs['medium'])

# Test configuration suggestions
complexities = ['low', 'medium', 'high']

print("Optimal Configuration Suggestions:")
for complexity in complexities:
    config = suggest_optimal_config(complexity)
    print(f"\n{complexity.upper()} Complexity:")
    print(f"  Temperature: {config['temperature']}")
    print(f"  Max Tokens: {config['max_tokens']}")
    print(f"  Thinking Budget: {config['thinking_budget']}")
    print(f"  Description: {config['description']}")

## Experiment Summary

In [ ]:
def create_experiment_summary():
    """Create a summary of experimentation findings."""
    
    summary = {
        'key_findings': [
            'Lower temperatures (0.1-0.3) produce more consistent code',
            'Higher temperatures (0.7-1.0) increase creativity but may reduce accuracy',
            'Detailed prompts with examples improve code quality',
            'Step-by-step prompts work well for complex algorithms',
            'Including constraints helps focus the output'
        ],
        'recommendations': [
            'Use temperature 0.3 for production code generation',
            'Include examples in prompts for better structure',
            'Set appropriate token limits based on task complexity',
            'Monitor token usage to optimize costs',
            'Implement quality validation in post-processing'
        ],
        'next_experiments': [
            'Test different model versions',
            'Experiment with few-shot learning',
            'Compare performance across programming languages',
            'Test with real-world coding scenarios',
            'Evaluate code execution success rates'
        ]
    }
    
    return summary

# Display experiment summary
summary = create_experiment_summary()

print("EXPERIMENT SUMMARY")
print("=" * 50)

print("\n🔍 Key Findings:")
for finding in summary['key_findings']:
    print(f"  • {finding}")

print("\n💡 Recommendations:")
for rec in summary['recommendations']:
    print(f"  • {rec}")

print("\n🚀 Next Experiments:")
for exp in summary['next_experiments']:
    print(f"  • {exp}")